### Databricks notebook – Data generation code

In [0]:
%sql
Create catalog if not exists e_commerce;
create volume if not exists e_commerce.default.raw_data;

In [0]:
import random
import uuid
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType

# ============================================================
# 1. CONFIGURATION
# ============================================================

random.seed(42)

BASE_PATH = "/Volumes/e_commerce/default/raw_data"

BATCH_1 = "batch_001"
BATCH_2 = "batch_002"

# Change these values if required
TOTAL_CUSTOMERS = 500
TOTAL_ORDERS = 1000

# ============================================================
# 2. MASTER DATA
# ============================================================

first_names = [
    "Aarav", "Vivaan", "Aditya", "Arjun", "Rahul",
    "Priya", "Ananya", "Isha", "Sneha", "Neha",
    "Rohan", "Karan", "Simran", "Meera", "Amit",
    "Kavya", "Riya", "Vikram", "Pooja", "Manish"
]

last_names = [
    "Sharma", "Verma", "Singh", "Gupta", "Negi",
    "Mehta", "Kumar", "Joshi", "Malhotra", "Reddy",
    "Patel", "Das", "Kapoor", "Yadav", "Chauhan"
]

cities = [
    "Delhi", "Mumbai", "Bangalore", "Hyderabad",
    "Pune", "Jaipur", "Lucknow", "Dehradun",
    "Chandigarh", "Noida", "Gurgaon", "Kolkata"
]

products = [
    ("P101", "Laptop", 55000),
    ("P102", "Mobile", 25000),
    ("P103", "Headphones", 2500),
    ("P104", "Smart Watch", 5000),
    ("P105", "Tablet", 18000),
    ("P106", "Keyboard", 1500),
    ("P107", "Monitor", 12000),
    ("P108", "Mouse", 800),
    ("P109", "Camera", 35000),
    ("P110", "Speaker", 3000)
]

payment_methods = [
    "UPI", "Credit Card", "Debit Card",
    "Net Banking", "COD", "Wallet"
]

order_statuses = [
    "PLACED", "CONFIRMED", "SHIPPED",
    "OUT_FOR_DELIVERY", "DELIVERED", "CANCELLED"
]

# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def random_date(start_date, end_date):
    days = (end_date - start_date).days
    return start_date + timedelta(days=random.randint(0, days))


def maybe_null(value, probability=0.05):
    return None if random.random() < probability else value


def make_customer(customer_id, registration_date):
    first = random.choice(first_names)
    last = random.choice(last_names)

    return {
        "customer_id": customer_id,
        "first_name": first,
        "last_name": last,
        "email": f"{first.lower()}.{last.lower()}.{customer_id}@gmail.com",
        "phone": f"9{random.randint(100000000, 999999999)}",
        "city": random.choice(cities),
        "gender": random.choice(["Male", "Female", "Other"]),
        "registration_date": registration_date.strftime("%Y-%m-%d"),
        "customer_segment": random.choice(
            ["Regular", "Premium", "VIP"]
        ),
        "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }


def write_csv(rows, dataset, batch):
    path = f"{BASE_PATH}/{dataset}/{batch}"

    if not rows:
        print(f"No records to write: {path}")
        return

    schema = StructType([
        StructField(column, StringType(), True)
        for column in rows[0].keys()
    ])

    df = spark.createDataFrame(rows, schema=schema)

    (df.write
       .format("csv")
       .mode("overwrite")
       .option("header", True)
       .save(path))

    print(f"{dataset.upper()} | {len(rows)} records | {path}")


# ============================================================
# 4. GENERATE CUSTOMER DATA - BATCH 001
# ============================================================

customer_master = {}

customer_batch_1 = []

for i in range(1, TOTAL_CUSTOMERS + 1):

    customer_id = f"CUST{i:05d}"

    registration_date = random_date(
        datetime(2023, 1, 1),
        datetime(2026, 10, 31)
    )

    customer = make_customer(customer_id, registration_date)

    customer_master[customer_id] = customer

    row = customer.copy()

    # Introduce NULL values
    row["email"] = maybe_null(row["email"], 0.05)
    row["phone"] = maybe_null(row["phone"], 0.04)
    row["city"] = maybe_null(row["city"], 0.03)

    # Introduce some invalid emails
    if random.random() < 0.02:
        row["email"] = "invalid_email"

    customer_batch_1.append(row)

# Add a duplicate record for data quality testing
customer_batch_1.append(customer_batch_1[0].copy())


# ============================================================
# 5. GENERATE ORDER DATA - BATCH 001
# ============================================================

order_master = {}

order_batch_1 = []

for i in range(1, TOTAL_ORDERS + 1):

    order_id = f"ORD{i:06d}"

    customer_id = random.choice(list(customer_master.keys()))

    product_id, product_name, price = random.choice(products)

    quantity = random.randint(1, 5)

    order_date = random_date(
        datetime(2025, 1, 1),
        datetime(2026, 11, 15)
    )

    order = {
        "order_id": order_id,
        "customer_id": customer_id,
        "product_id": product_id,
        "product_name": product_name,
        "quantity": str(quantity),
        "unit_price": str(price),
        "total_amount": str(quantity * price),
        "payment_method": random.choice(payment_methods),
        "order_date": order_date.strftime("%Y-%m-%d"),
        "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    order_master[order_id] = order
    row = order.copy()

    # Introduce NULL values
    row["payment_method"] = maybe_null(row["payment_method"], 0.04)
    row["product_name"] = maybe_null(row["product_name"], 0.02)

    order_batch_1.append(row)

# Add a duplicate order
order_batch_1.append(order_batch_1[0].copy())


# ============================================================
# 6. GENERATE STATUS HISTORY - BATCH 001
# ============================================================

status_batch_1 = []

status_sequence = [
    "PLACED",
    "CONFIRMED",
    "SHIPPED",
    "OUT_FOR_DELIVERY",
    "DELIVERED"
]

for order in order_master.values():

    order_id = order["order_id"]

    order_date = datetime.strptime(
        order["order_date"], "%Y-%m-%d"
    )

    # Some orders will be cancelled
    if random.random() < 0.12:
        statuses = ["PLACED", "CANCELLED"]
    else:
        final_index = random.randint(1, 4)
        statuses = status_sequence[:final_index + 1]

    for sequence, status in enumerate(statuses, start=1):

        status_date = order_date + timedelta(days=sequence - 1)

        status_batch_1.append({
            "status_id": f"STS{uuid.uuid4().hex[:10].upper()}",
            "order_id": order_id,
            "status": status,
            "status_sequence": str(sequence),
            "status_date": status_date.strftime("%Y-%m-%d"),
            "status_description": maybe_null(
                f"Order {status.lower().replace('_', ' ')}",
                0.05
            ),
            "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })


# ============================================================
# 7. GENERATE CUSTOMER CHANGES - BATCH 002
#    Required for SCD TYPE 2
# ============================================================

customer_batch_2 = []

existing_customer_ids = random.sample(
    list(customer_master.keys()), 50
)

for customer_id in existing_customer_ids:

    old_customer = customer_master[customer_id]

    updated_customer = old_customer.copy()

    # Change customer attributes
    updated_customer["city"] = random.choice(
        [city for city in cities if city != old_customer["city"]]
    )

    updated_customer["customer_segment"] = random.choice(
        ["Regular", "Premium", "VIP"]
    )

    updated_customer["email"] = (
        f"updated.{customer_id.lower()}@gmail.com"
    )

    updated_customer["updated_at"] = (
        datetime.now() + timedelta(days=1)
    ).strftime("%Y-%m-%d %H:%M:%S")

    customer_master[customer_id] = updated_customer

    row = updated_customer.copy()

    row["email"] = maybe_null(row["email"], 0.03)

    customer_batch_2.append(row)


# Generate 50 NEW customers

new_customer_start = TOTAL_CUSTOMERS + 1

for i in range(new_customer_start, new_customer_start + 50):

    customer_id = f"CUST{i:05d}"

    registration_date = random_date(
        datetime(2026, 1, 1),
        datetime(2026, 11, 15)
    )

    customer = make_customer(customer_id, registration_date)

    customer_master[customer_id] = customer

    row = customer.copy()
    row["phone"] = maybe_null(row["phone"], 0.05)

    customer_batch_2.append(row)


# ============================================================
# 8. GENERATE NEW ORDERS - BATCH 002
# ============================================================

order_batch_2 = []

new_order_start = TOTAL_ORDERS + 1

for i in range(new_order_start, new_order_start + 200):

    order_id = f"ORD{i:06d}"

    customer_id = random.choice(list(customer_master.keys()))

    product_id, product_name, price = random.choice(products)

    quantity = random.randint(1, 5)

    order = {
        "order_id": order_id,
        "customer_id": customer_id,
        "product_id": product_id,
        "product_name": product_name,
        "quantity": str(quantity),
        "unit_price": str(price),
        "total_amount": str(quantity * price),
        "payment_method": random.choice(payment_methods),
        "order_date": random_date(
            datetime(2026, 11, 1),
            datetime(2026, 11, 22)
        ).strftime("%Y-%m-%d"),
        "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    order_master[order_id] = order

    row = order.copy()
    row["payment_method"] = maybe_null(row["payment_method"], 0.04)

    order_batch_2.append(row)


# ============================================================
# 9. GENERATE NEW STATUS EVENTS - BATCH 002
# ============================================================

status_batch_2 = []

# Add status events for newly created orders
for order in order_batch_2:

    order_date = datetime.strptime(
        order["order_date"], "%Y-%m-%d"
    )

    for sequence, status in enumerate(
        ["PLACED", "CONFIRMED"], start=1
    ):

        status_batch_2.append({
            "status_id": f"STS{uuid.uuid4().hex[:10].upper()}",
            "order_id": order["order_id"],
            "status": status,
            "status_sequence": str(sequence),
            "status_date": (
                order_date + timedelta(days=sequence - 1)
            ).strftime("%Y-%m-%d"),
            "status_description": f"Order {status.lower()}",
            "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })


# Add new status events for 100 existing orders
existing_order_ids = random.sample(
    list(order_master.keys())[:TOTAL_ORDERS], 100
)

for order_id in existing_order_ids:

    order_date = datetime.strptime(
        order_master[order_id]["order_date"], "%Y-%m-%d"
    )

    status_batch_2.append({
        "status_id": f"STS{uuid.uuid4().hex[:10].upper()}",
        "order_id": order_id,
        "status": "SHIPPED",
        "status_sequence": "3",
        "status_date": (
            order_date + timedelta(days=2)
        ).strftime("%Y-%m-%d"),
        "status_description": "Order shipped",
        "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })


# ============================================================
# 10. WRITE ALL DATA INTO DATABRICKS VOLUMES
# ============================================================

write_csv(customer_batch_1, "customer", BATCH_1)
write_csv(order_batch_1, "orders", BATCH_1)
write_csv(status_batch_1, "status", BATCH_1)

write_csv(customer_batch_2, "customer", BATCH_2)
write_csv(order_batch_2, "orders", BATCH_2)
write_csv(status_batch_2, "status", BATCH_2)

print("\n====================================")
print("E-COMMERCE DATA GENERATION COMPLETED")
print("====================================")
